# generate_onegrid_data (deploy) — build og.* unified tables
Writes the conformed OneGrid model tables to the lakehouse `og` schema via abfss. PiEvents disabled for deploy.

In [ ]:
# Fabric notebook: generate_onegrid_data
# =====================================================================================
# OneGrid unified synthetic data generator (see ../MODEL.md).
# Produces ONE conformed asset model (dim_site -> dim_unit -> dim_asset -> dim_tag) with
# TWO fact domains over it:
#   A) Reliability / condition (digital twin): aakr_health, watchlist, anomaly_advisories,
#      root_cause, predictions_shortterm, predictions_longterm, fact_work_requests, daily_narrative
#   B) Weather / exposure: dim_weather_event, fact_asset_exposure, fact_posture
#   + time-series: PiEvents (tiered historian), WeatherForecast, PCIOutages
#
# Everything except PiEvents is tiny and generated deterministically in Python, then written
# as Delta. PiEvents is the only big table and is generated Spark-native.
#
# Runs in a Fabric notebook attached to a Lakehouse (schemas enabled). Cells are delimited
# with `# CELL ****`. HURDAT2 real tracks are read from ../reference/hurdat2_gulf.csv
# (upload it to the Lakehouse Files/ area, or point HURDAT_PATH at it).
# =====================================================================================

# CELL ********************  (parameters)
# Tag this cell as "Parameters" in Fabric so these can be overridden per run.
SCHEMA        = "og"
SEED          = 42
HISTORY_DAYS  = 30
HOT_DAYS      = 1
PIEVENTS_ENABLED = False      # deploy: skip the huge historian (not part of OneGridModel)
WRITE_MODE    = "overwrite"
# --- deploy rebind: these GUIDs are find/replaced with the target ws/lakehouse ---
WS = "163ba38c-3869-406f-adb7-37cbc981390c"
LH = "7e08480c-cf8d-4206-901d-38b74dbe35d9"
BASE   = f"abfss://{WS}@onelake.dfs.fabric.microsoft.com/{LH}"
TABLES = f"{BASE}/Tables"
HURDAT_PATH = f"{BASE}/Files/reference/hurdat2_gulf.csv"

# CELL ********************  (imports, spark, helpers)
import math, hashlib, datetime as dt
from pyspark.sql import functions as F, types as T

spark.conf.set("spark.sql.session.timeZone", "UTC")
try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
except Exception as _e:
    print("schema pre-create skipped (created implicitly by abfss writes):", _e)

def h32(*parts):
    """Deterministic 32-bit hash of the given parts."""
    s = "|".join(str(p) for p in parts)
    return int(hashlib.blake2b(s.encode(), digest_size=4).hexdigest(), 16)

def rng01(*parts):
    """Deterministic pseudo-random in [0,1)."""
    return (h32(*parts) % 1_000_000) / 1_000_000.0

EARTH_MI = 3958.8
def haversine_mi(a_lat, a_lon, b_lat, b_lon):
    dlat = math.radians(b_lat - a_lat); dlon = math.radians(b_lon - a_lon)
    x = math.sin(dlat/2)**2 + math.cos(math.radians(a_lat))*math.cos(math.radians(b_lat))*math.sin(dlon/2)**2
    return 2*EARTH_MI*math.asin(math.sqrt(min(1, max(0, x))))

def saffir(wind_mph):
    kt = wind_mph/1.15078
    return 5 if kt>=137 else 4 if kt>=113 else 3 if kt>=96 else 2 if kt>=83 else 1 if kt>=64 else 0

# NHC 5-yr average track-error cone radii (nautical miles) by forecast hour -> miles.
CONE_NM = {12:26, 24:40, 36:51, 48:66, 60:79, 72:93, 96:116, 120:144}
def cone_mi(hour):
    keys = sorted(CONE_NM)
    if hour <= keys[0]: return CONE_NM[keys[0]]*1.15078
    if hour >= keys[-1]: return CONE_NM[keys[-1]]*1.15078
    for i in range(len(keys)-1):
        a,b = keys[i], keys[i+1]
        if a <= hour <= b:
            f = (hour-a)/(b-a)
            return (CONE_NM[a] + (CONE_NM[b]-CONE_NM[a])*f)*1.15078
    return CONE_NM[keys[-1]]*1.15078

NOW = dt.datetime.utcnow().replace(microsecond=0)

# CELL ********************  (site + equipment configuration — 15 sites, see MODEL.md §6)
# (site_id, name, site_type, region, lat, lon, operator, business_unit, criticality)
SITES = [
    ("SITE_RV","Riverton","power_plant","mountain_west",43.0,-108.4,"OGE Power","Thermal","important"),
    ("SITE_FV","Fairview","power_plant","south",33.1,-96.6,"OGE Power","Thermal","business_critical"),
    ("SITE_AS","Ashford","power_plant","west_coast",46.9,-122.0,"OGE Power","Thermal","important"),
    ("SITE_DW","Deepwater","power_plant","northeast",39.68,-75.48,"OGE Power","Thermal","important"),
    ("SITE_HP","Harbor Point","power_plant","midwest",41.8,-87.6,"OGE Power","Thermal","business_critical"),
    ("SITE_CF","Cedar Falls","power_plant","midwest",42.53,-92.44,"OGE Power","Thermal","standard"),
    ("SITE_GW","Glenwood","power_plant","midwest",45.65,-95.39,"OGE Power","Thermal","standard"),
    ("SITE_BK","Brookline","power_plant","northeast",42.33,-71.12,"OGE Power","Thermal","standard"),
    ("SITE_TH","Thunder Horse","offshore_platform","gulf_offshore",28.2,-88.5,"OGE Upstream","Offshore","business_critical"),
    ("SITE_MD","Mad Dog","offshore_platform","gulf_offshore",27.2,-90.3,"OGE Upstream","Offshore","business_critical"),
    ("SITE_AT","Atlantis","offshore_platform","gulf_offshore",27.2,-90.0,"OGE Upstream","Offshore","business_critical"),
    ("SITE_PF","Port Fourchon","terminal","gulf_coast",29.1,-90.2,"OGE Midstream","Logistics","important"),
    ("SITE_CA","Cascadia LNG","lng","west_coast",46.2,-123.9,"OGE Midstream","LNG","important"),
    ("SITE_PR","Permian Ridge","pipeline","south",31.8,-102.4,"OGE Midstream","Pipeline","standard"),
    ("SITE_EP","Eastport","power_plant","northeast",44.9,-67.0,"OGE Power","Thermal","standard"),
]

# equipment archetypes by site_type: list of (unit_type, [ (equipment_name, category, group,
#   [ (tag_stem, descriptor, units) ]) ])
def train(u_type, items): return (u_type, items)
def eq(name, cat, grp, tags): return (name, cat, grp, tags)

ARCHETYPES = {
    "power_plant": [
        train("steam_train", [
            eq("Boiler","Fired","boiler",[("BLR.DRUM_PRESS","Drum pressure","bar"),("BLR.STEAM_TEMP","Main steam temp","degC"),("BLR.O2","Flue-gas O2","%")]),
            eq("Steam Turbine","Rotating","turbine",[("STM.VIBR","Turbine vibration","mm/s"),("STM.RPM","Turbine speed","rpm"),("STM.BRG_TEMP","Bearing metal temp","degC")]),
            eq("Boiler Feed Pump","Rotating","pump",[("BFP.VIBR","Feed pump vibration","mm/s"),("BFP.BRG_TEMP","Feed pump bearing temp","degC")]),
            eq("Generator","Electrical","generator",[("GEN.STATOR_TEMP","Stator winding temp","degC"),("GEN.MW","Net load","MW")]),
        ]),
    ],
    "offshore_platform": [
        train("compression", [
            eq("Gas Compressor","Rotating","compressor",[("GC.VIBR","Compressor vibration","mm/s"),("GC.DISCH_PRESS","Discharge pressure","bar"),("GC.BRG_TEMP","Bearing temp","degC")]),
            eq("Compressor Driver","Electrical","generator",[("GCD.STATOR_TEMP","Driver stator temp","degC"),("GCD.MW","Driver load","MW")]),
        ]),
        train("pumping", [
            eq("Export Pump","Rotating","pump",[("XP.VIBR","Export pump vibration","mm/s"),("XP.DISCH_PRESS","Discharge pressure","bar")]),
            eq("Separator","Static","separator",[("SEP.LVL","Separator level","%"),("SEP.PRESS","Separator pressure","bar")]),
        ]),
    ],
    "well": [
        train("wellhead", [
            eq("Wellhead","Static","wellhead",[("WH.PRESS","Wellhead pressure","bar"),("WH.TEMP","Wellhead temp","degC")]),
            eq("ESP","Rotating","pump",[("ESP.VIBR","ESP vibration","mm/s"),("ESP.MOTOR_TEMP","ESP motor temp","degC")]),
        ]),
    ],
    "terminal": [
        train("pumping", [
            eq("Loading Pump","Rotating","pump",[("LP.VIBR","Loading pump vibration","mm/s"),("LP.DISCH_PRESS","Discharge pressure","bar")]),
            eq("Metering Skid","Static","separator",[("MET.FLOW","Metered flow","m3/h"),("MET.PRESS","Line pressure","bar")]),
        ]),
    ],
    "lng": [
        train("compression", [
            eq("Boil-off Compressor","Rotating","compressor",[("BOG.VIBR","BOG compressor vibration","mm/s"),("BOG.DISCH_PRESS","Discharge pressure","bar")]),
            eq("Loading Pump","Rotating","pump",[("LNGP.VIBR","LNG pump vibration","mm/s"),("LNGP.MOTOR_TEMP","Pump motor temp","degC")]),
        ]),
    ],
    "pipeline": [
        train("compression", [
            eq("Compressor Station","Rotating","compressor",[("CS.VIBR","Station vibration","mm/s"),("CS.DISCH_PRESS","Discharge pressure","bar"),("CS.BRG_TEMP","Bearing temp","degC")]),
            eq("Metering Skid","Static","separator",[("PMET.FLOW","Metered flow","m3/h"),("PMET.PRESS","Line pressure","bar")]),
        ]),
    ],
}
# unit code prefix per site (used as PI tag prefix), e.g. RV3
SITE_PREFIX = {"SITE_RV":"RV","SITE_FV":"FV","SITE_AS":"AS","SITE_DW":"DW","SITE_HP":"HP","SITE_CF":"CF",
    "SITE_GW":"GW","SITE_BK":"BK","SITE_TH":"TH","SITE_MD":"MD","SITE_AT":"AT","SITE_PF":"PF",
    "SITE_CA":"CA","SITE_PR":"PR","SITE_EP":"EP"}

# seeded condition profile: a handful of sites carry degraded equipment
CRITICAL = {"FV_U1_Steam_Turbine","HP_U1_Boiler","TH_U1_Gas_Compressor","MD_U2_Export_Pump"}
WATCH    = {"CF_U1_Boiler_Feed_Pump","DW_U1_Steam_Turbine","AS_U1_Generator","PR_U1_Compressor_Station","CA_U1_Boil-off_Compressor"}

# CELL ********************  (build dims: dim_site, dim_unit, dim_asset, dim_tag)
def baseline(tag):
    t = tag.upper()
    if "MW" in t or "NLOAD" in t: mean,sd = 250+ h32(tag)%300, 8
    elif "VIBR" in t: mean,sd = 3.2 + h32(tag)%5, 0.55
    elif "STATOR" in t or "BRG_TEMP" in t or "MOTOR_TEMP" in t: mean,sd = 68 + h32(tag)%38, 2.6
    elif "STEAM_TEMP" in t: mean,sd = 538 + h32(tag)%12, 3.2
    elif "TEMP" in t: mean,sd = 60 + h32(tag)%40, 2.4
    elif "PRESS" in t: mean,sd = 40 + h32(tag)%60, 1.7
    elif "RPM" in t: mean,sd = 3000 + h32(tag)%620, 14
    elif "O2" in t: mean,sd = 3.2 + h32(tag)%2, 0.25
    elif "FLOW" in t: mean,sd = 400 + h32(tag)%600, 12
    elif "LVL" in t: mean,sd = 55 + h32(tag)%20, 3
    else: mean,sd = 50 + h32(tag)%40, 2
    return float(mean), float(sd)

sites, units, assets, tags = [], [], [], []
for (sid,name,stype,region,lat,lon,op,bu,crit) in SITES:
    sites.append((sid,name,stype,region,float(lat),float(lon),op,bu,crit))
    pfx = SITE_PREFIX[sid]
    archs = ARCHETYPES[stype]
    n_units = 2 if stype in ("power_plant","offshore_platform") else 1
    for u in range(1, n_units+1):
        # for platforms, alternate the two archetype trains; for others use train 0
        arch = archs[(u-1) % len(archs)]
        u_type, items = arch
        unit_code = f"{pfx}{u}"                 # e.g. FV1
        unit_id = f"{sid}_U{u}"
        units.append((unit_id, sid, unit_code, u_type))
        for (ename, cat, grp, tag_stems) in items:
            asset_id = f"{pfx}_U{u}_{ename.replace(' ','_')}"
            tag_list = [f"{unit_code}:{stem}" for (stem,_,_) in tag_stems]
            running = tag_list[0]
            assets.append((asset_id, sid, unit_id, ename, cat, grp, running))
            for (stem,desc,eu),tg in zip(tag_stems, tag_list):
                role = "watch" if ("VIBR" in stem or "TEMP" in stem or "PRESS" in stem) else "root cause"
                tags.append((tg, asset_id, desc, eu, role))

dim_site  = spark.createDataFrame(sites,  ["site_id","site_name","site_type","region","lat","lon","operator","business_unit","criticality"])
dim_unit  = spark.createDataFrame(units,  ["unit_id","site_id","unit_name","unit_type"])
dim_asset = spark.createDataFrame(assets, ["asset_id","site_id","unit_id","asset_display_name","equipment_category","equipment_group","running_tag"])
dim_tag   = spark.createDataFrame(tags,   ["tag","asset_id","descriptor","engineering_units","role"])
print(f"sites={len(sites)} units={len(units)} assets={len(assets)} tags={len(tags)}")

# CELL ********************  (Domain A — condition facts, deterministic per asset)
RC = {
 "turbine":("Outboard bearing wear","Rising 1x synchronous vibration coupled with bearing-metal temperature indicates progressing babbitt wear on the outboard journal bearing.","Inspect outboard bearing at the next window; stage a spare journal bearing.","VIBR","BRG_TEMP"),
 "pump":("Mechanical seal degradation","Seal-flush temperature drift and rising vibration point to a degrading mechanical seal face.","Schedule seal inspection; verify flush-line dP and top up barrier fluid.","VIBR","PRESS"),
 "boiler":("Drum-pressure control instability","Drum-pressure oscillation beyond the anomaly band with steam-temperature coupling suggests a sticking feedwater control valve.","Review level-control tuning and stroke the feedwater control valve.","PRESS","STEAM_TEMP"),
 "generator":("Stator cooling anomaly","Stator-winding temperature rising against stable load indicates reduced cooling effectiveness.","Check H2 purity and coolant differential; inspect stator cooler.","STATOR","MW"),
 "compressor":("Compressor bearing / surge margin","Rising vibration with discharge-pressure oscillation indicates reduced surge margin and bearing distress.","Check anti-surge control and bearing condition.","VIBR","PRESS"),
 "separator":("Level control drift","Level oscillation beyond band suggests level-control valve drift.","Calibrate level transmitter; stroke the LCV.","LVL","PRESS"),
 "wellhead":("Wellhead pressure decline","Gradual pressure decline vs baseline indicates reservoir/completion change.","Review well test; schedule intervention.","PRESS","TEMP"),
}
def status_of(asset_id):
    return "critical" if asset_id in CRITICAL else "watch" if asset_id in WATCH else "ok"

def profile(asset_id):
    r = rng01(asset_id,"prof"); st = status_of(asset_id)
    if st=="critical": return dict(status=st, maxZ=6+r*3, stop=0.55+r*0.2, surv14=0.36+r*0.18, surv7=0.6+r*0.12, risk=0.72+r*0.2, risk_level="critical", health=46+r*12)
    if st=="watch":    return dict(status=st, maxZ=3+r*2, stop=0.2+r*0.15, surv14=0.7+r*0.12, surv7=0.82+r*0.1, risk=0.4+r*0.25, risk_level="high" if r>0.5 else "medium", health=68+r*10)
    return dict(status=st, maxZ=r*2, stop=0.02+r*0.1, surv14=0.9+r*0.09, surv7=0.95+r*0.04, risk=r*0.2, risk_level="low", health=84+r*14)

asset_meta = {a[0]: dict(site_id=a[1], unit_id=a[2], name=a[3], group=a[5], tags=[t[0] for t in tags if t[1]==a[0]]) for a in assets}
site_by_id = {s[0]: s for s in sites}
unit_by_id = {u[0]: u for u in units}
tag_desc   = {t[0]: t[2] for t in tags}

health_rows, watch_rows, anom_rows, rc_rows, ps_rows, pl_rows = [], [], [], [], [], []
ts_now = NOW.isoformat()+"Z"
for aid, m in asset_meta.items():
    pr = profile(aid); grp = m["group"]; atags = m["tags"]
    health = round(pr["health"],1)
    health_rows.append((aid, health, round(pr["maxZ"]*3,1), round(pr["maxZ"],2)))  # aakr_health: health_score, anomaly_pct, max_abs_z
    rc = RC.get(grp, RC["turbine"])
    prim = next((t for t in atags if rc[3] in t.upper()), atags[0])
    contrib = next((t for t in atags if rc[4] in t.upper()), atags[-1])
    ndeg = 3 if pr["status"]=="critical" else 2 if pr["status"]=="watch" else 0
    # watchlist (rich)
    for i,t in enumerate(atags[: (1 if pr["status"]=="ok" else ndeg+1)]):
        b_mean,b_sd = baseline(t); up = ("O2" not in t and "PRESS" not in t) or i%2==0
        slope = (1 if pr["status"]=="critical" else 0.5 if pr["status"]=="watch" else 0.12)*b_sd*(1 if up else -1)*(1+rng01(t,"w"))
        cur = round(b_mean + slope*2 + (rng01(t,"cur")-0.5)*b_sd, 2)
        watch_rows.append((t, aid, tag_desc.get(t, t), "increasing" if up else "decreasing",
                           cur, round(b_mean,2), round(b_mean-4*b_sd,2), round(b_mean+4*b_sd,2), round(slope,3),
                           round((0.7 if pr["status"]=="critical" else 0.4 if pr["status"]=="watch" else 0.1)*(1+rng01(t,"rc"))-i*0.05,2),
                           rc[2] if i==0 else ""))
    # anomalies
    if pr["status"]!="ok":
        for i,t in enumerate(atags[:ndeg]):
            z = round(pr["maxZ"]-i*0.8+rng01(t,"z")*0.4,1); b_mean,b_sd = baseline(t)
            sev = "CRITICAL" if z>=6 else "HIGH" if z>=3 else "MEDIUM"
            anom_rows.append((t, aid, sev, z, round(2+rng01(t,"d")*20,1),
                              f"{t} deviating {z} sigma from AAKR baseline.", round(b_mean+z*b_sd*0.4,2), round(b_mean,2),
                              "low" if "O2" in t else "high"))
    # root cause
    if pr["status"]!="ok":
        rc_rows.append((aid, prim, rc[0], rc[1], "Critical" if pr["status"]=="critical" else "Medium",
                        round(0.82+rng01(aid,"c")*0.12 if pr["status"]=="critical" else 0.62+rng01(aid,"c")*0.15,2),
                        rc[2], contrib))
    # predictions
    if pr["status"]!="ok":
        for i,hz in enumerate(["4h","8h","12h","24h","48h","72h"]):
            ps_rows.append((aid, hz, round(min(0.95,max(0.01,pr["stop"]*(0.5+i*0.12))),3),
                            "Critical" if (pr["stop"]>=0.55 and i>=3) else "High" if pr["stop"]>=0.35 else "Medium", ts_now))
    md = round(6+rng01(aid,"m")*6 if pr["surv14"]<0.5 else 40+rng01(aid,"m")*60,0)
    for hz in ["7d","14d"]:
        pl_rows.append((aid, hz, pr["risk_level"], round(pr["risk"],3), round(pr["surv7"],3), round(pr["surv14"],3), md))

aakr_health = spark.createDataFrame(health_rows, ["asset_id","health_score","anomaly_pct","max_abs_z"])
watchlist   = spark.createDataFrame(watch_rows, ["tag_name","asset_id","descriptor","trend_direction","current_value","baseline_mean","normal_range_low","normal_range_high","trend_slope_per_day","risk_contribution","recommended_action"])
anomaly_advisories = spark.createDataFrame(anom_rows, ["Tag","asset_id","severity","peak_abs_z","duration_h","advisory_message","latest_value","baseline_median","anomaly_direction"])
root_cause  = spark.createDataFrame(rc_rows, ["asset_id","tag","failure_mechanism","root_cause","priority","confidence","recommended_action","contributing_tag_names"])
predictions_shortterm = spark.createDataFrame(ps_rows, ["asset_id","prediction_horizon","stop_probability","alert_level","scoring_timestamp"])
predictions_longterm  = spark.createDataFrame(pl_rows, ["asset_id","horizon","risk_level","risk_score","survival_probability_7d","survival_probability_14d","predicted_median_survival_days"])
print(f"health={len(health_rows)} watch={len(watch_rows)} anom={len(anom_rows)} rc={len(rc_rows)} predS={len(ps_rows)} predL={len(pl_rows)}")

# CELL ********************  (work orders + daily narrative)
WR_TYPES=["Corrective","Preventive","Inspection","Predictive"]; WR_STATUS=["In Progress","Scheduled","Ready to Schedule","New Request","Planning Required"]
wo_rows=[]; n=48210
PROB={"turbine":["Bearing vibration alarm — investigate outboard journal","Governor valve stroke test overdue"],
      "pump":["Mechanical seal leak","Bearing temperature trending high"],
      "boiler":["Drum level control instability","Flue-gas O2 analyzer drift"],
      "generator":["Stator temperature high — inspect cooling","H2 purity low"],
      "compressor":["Compressor vibration high","Anti-surge valve fault"],
      "separator":["Level control drift","LCV sticking"],"wellhead":["Wellhead pressure decline","Valve leak-by"]}
for aid,m in asset_meta.items():
    st=status_of(aid); cnt=3 if st=="critical" else 2 if st=="watch" else (1 if rng01(aid,"wo")>0.6 else 0)
    site=site_by_id[m["site_id"]]; unit=unit_by_id[m["unit_id"]]; probs=PROB.get(m["group"],PROB["turbine"])
    for i in range(cnt):
        prio = 1 if (st=="critical" and i==0) else 2 if st=="watch" else 2+int(rng01(aid,"p",i)*3)
        wo_rows.append((f"WR-{n}", probs[i%len(probs)], f"{site[1]} · {unit[2]}", f"{unit[2]} {m['name']}", m['name'],
                        WR_TYPES[int(rng01(aid,'t',i)*4)], WR_STATUS[int(rng01(aid,'s',i)*len(WR_STATUS))], "OPEN",
                        prio, prio, site[1], aid,
                        (NOW - dt.timedelta(days=rng01(aid,'cd',i)*30)).isoformat()+"Z")); n+=1
fact_work_requests = spark.createDataFrame(wo_rows, ["wr_id","problem_descr","location","parent_descr","entity_descr","wr_type","wr_status","wr_status_id","priority","priority_rank","site_name","asset_id","create_date"])

brief=[]
for aid,m in asset_meta.items():
    if status_of(aid)!="ok":
        rc=RC.get(m["group"],RC["turbine"])
        brief.append(f"[{'CRITICAL' if status_of(aid)=='critical' else 'MONITOR'}] {unit_by_id[m['unit_id']][2]} {m['name']}: {rc[0]} — {rc[2].split(';')[0]}")
daily_narrative = spark.createDataFrame([(NOW.date().isoformat(), " | ".join(brief))], ["narrative_date","narrative_text"])
print(f"work_orders={len(wo_rows)}")

# CELL ********************  (Domain B — weather events from REAL HURDAT tracks + synthesized hazards)
hurdat = spark.read.option("header",True).csv(HURDAT_PATH).toPandas()
hurdat["lat"]=hurdat["lat"].astype(float); hurdat["lon"]=hurdat["lon"].astype(float); hurdat["wind_mph"]=hurdat["wind_mph"].astype(float)

# Pick one storm as the ACTIVE incoming hurricane (Ida-like Gulf landfall). Re-anchor its
# track so "now" (t0) sits ~18h before Gulf entry, and expose forward points as the forecast.
active_storm = "AL092021"  # Ida 2021
strk = hurdat[hurdat.storm_id==active_storm].reset_index(drop=True)
# choose t0 index where storm enters the Gulf (lat<27, lon between -95 and -84)
gulf_idx = strk.index[(strk.lat<27)&(strk.lon>-95)&(strk.lon<-84)]
t0 = int(gulf_idx[0]) if len(gulf_idx) else max(0, len(strk)-16)
event_rows=[]; forecast_rows=[]
def add_event(eid, kind, name, region, status, cat, wind, gust, pres, lat, lon, mvdeg, mvmph):
    event_rows.append((eid, kind, name, region, status, cat, wind, gust, pres, lat, lon, mvdeg, mvmph, NOW.isoformat()+"Z"))

# hurricane forecast points (hour 0..120 from t0)
cur = strk.iloc[t0]
add_event("EVT_HUR","hurricane","Hurricane (Ida track)","gulf_offshore","active", saffir(cur.wind_mph),
          round(cur.wind_mph), round(cur.wind_mph*1.25), 950, float(cur.lat), float(cur.lon), 315, 12)
for j in range(t0, min(len(strk), t0+21)):  # 6h steps -> up to 120h
    p = strk.iloc[j]; hour=(j-t0)*6
    forecast_rows.append(("EVT_HUR", hour, float(p.lat), float(p.lon), float(p.wind_mph), round(cone_mi(hour),1), saffir(p.wind_mph), 950))

# synthesized Midwest severe-convective (tornado) outbreak — outlook centroid + radius
add_event("EVT_SVR","severe_convective","Plains Tornado Outbreak","midwest","active", 0, 70, 90, 995, 41.5,-90.0, 250, 30)
for hour in range(0,25,3):
    forecast_rows.append(("EVT_SVR", hour, 41.5+hour*0.12, -90.0+hour*0.25, 70.0, 120.0, 0, 995))  # radius as "cone" proxy
# synthesized West-coast high-wind event
add_event("EVT_WND","high_wind","Cascadia Wind Event","west_coast","active", 0, 65, 85, 990, 46.5,-123.5, 90, 25)
for hour in range(0,25,3):
    forecast_rows.append(("EVT_WND", hour, 46.5, -123.5+hour*0.2, 65.0, 150.0, 0, 990))

dim_weather_event = spark.createDataFrame(event_rows, ["event_id","hazard_kind","name","region","status","category","current_wind_mph","gust_mph","pressure_mb","lat","lon","movement_deg","movement_mph","updated_at"])
WeatherForecast   = spark.createDataFrame(forecast_rows, ["event_id","hour","lat","lon","wind_mph","cone_radius_mi","category","pressure_mb"])
fc_by_event = {}
for r in forecast_rows: fc_by_event.setdefault(r[0], []).append(r)
print(f"events={len(event_rows)} forecast_points={len(forecast_rows)} (active hurricane t0={t0})")

# CELL ********************  (exposure scoring — port of risk-engine.ts; small volume)
TYPE_SENS={"offshore_platform":8,"pipeline":5,"lng":6,"refinery":6,"terminal":5,"port":5,"power_plant":4,"well":2}
CRIT_PTS={"business_critical":10,"important":6,"standard":2}
# which hazards apply to which region
REGION_HAZ={"gulf_offshore":["EVT_HUR"],"gulf_coast":["EVT_HUR"],"south":["EVT_HUR","EVT_SVR"],
    "midwest":["EVT_SVR"],"mountain_west":["EVT_WND"],"west_coast":["EVT_WND"],"northeast":["EVT_HUR"]}
def wind_at(dist, core):
    if dist<=30: return float(core)
    return float(max(12, round(core*math.exp(-(dist-30)/95))))
def rain_at(dist, cat): return max(0.2, round((3+cat*1.9)*math.exp(-dist/180),1))
def level_from(s): return "critical" if s>=80 else "high" if s>=62 else "elevated" if s>=42 else "monitor" if s>=22 else "normal"

exp_rows=[]; posture_rows=[]
for aid,m in asset_meta.items():
    site=site_by_id[m["site_id"]]; slat,slon,region,stype,crit = site[4],site[5],site[3],site[2],site[8]
    for eid in REGION_HAZ.get(region, []):
        pts = fc_by_event.get(eid, [])
        if not pts: continue
        # closest approach (interpolate segments into 6ths)
        best=None
        for i in range(len(pts)):
            samples=[(pts[i][2],pts[i][3],pts[i][1],pts[i])]
            if i+1<len(pts):
                a,b=pts[i],pts[i+1]
                for t in range(1,6):
                    f=t/6; samples.append((a[2]+(b[2]-a[2])*f, a[3]+(b[3]-a[3])*f, a[1]+(b[1]-a[1])*f, a if f<0.5 else b))
            for (plat,plon,phour,base) in samples:
                d=haversine_mi(slat,slon,plat,plon)
                if best is None or d<best[0]: best=(d,phour,base)
        dist,hour,base = best; cat=base[6]; core=base[4]; cone=base[5]
        w=wind_at(dist,core); rn=rain_at(dist,cat)
        pts_sum = round(max(0,26*math.exp(-dist/70)))              # proximity
        pts_sum+= round(min(24,max(0,(w-50)/3.7)))                 # wind
        pts_sum+= round(min(8, rn*0.9))                            # rain
        pts_sum+= round(max(0,10-hour/12))                         # eta
        pts_sum+= 0 if dist>220 else round(cat*2)                  # intensity
        pts_sum+= CRIT_PTS[crit]                                   # criticality
        pts_sum+= TYPE_SENS.get(stype,4)                           # type sensitivity
        inside = dist<=max(cone,25)
        if inside: pts_sum+=6
        score=max(0,min(100,round(pts_sum))); lvl=level_from(score)
        threat = f"{w} mph sustained wind" if w>=74 else (f"EF-scale tornado risk" if eid=="EVT_SVR" and dist<150 else f"{rn} in rainfall" if rn>=4 else "monitoring only")
        haz = next(e[1] for e in event_rows if e[0]==eid)
        exp_rows.append((aid, m["site_id"], "18Z", haz, score, lvl, threat, float(w), float(rn), round(dist), int(round(hour)), inside))
    # posture for the site's worst exposure (offshore/coastal mainly)
# simplify posture: site-level from max exposure
from collections import defaultdict
site_max=defaultdict(int)
for r in exp_rows: site_max[r[1]]=max(site_max[r[1]], r[4])
for sid,score in site_max.items():
    lvl = 4 if score>=80 else 3 if score>=62 else 2 if score>=42 else 1 if score>=22 else 0
    pob_norm = 128 if site_by_id[sid][2]=="offshore_platform" else 0
    pob_cur = 0 if lvl>=4 else pob_norm
    posture_rows.append((sid, lvl, pob_cur, pob_norm, "GoM Offshore Installation Manager"))

# drop the messy placeholder col; rebuild exposure rows cleanly
fact_asset_exposure = spark.createDataFrame(exp_rows, ["asset_id","site_id","cycle_id","hazard_kind","score","level","primary_threat","forecast_wind_mph","rainfall_in","distance_mi","hours_to_impact","inside_threat_area"])
fact_posture = spark.createDataFrame(posture_rows, ["site_id","posture_level","pob_current","pob_normal","decision_owner"])
print(f"exposure_rows={len(exp_rows)} posture_sites={len(posture_rows)}")

# CELL ********************  (PiEvents — Spark-native tiered historian: cold 1-min 30d + hot 1-sec 1d)
if PIEVENTS_ENABLED:
    # small baseline table -> broadcast join
    base_rows=[]
    for (tg,aid,desc,eu,role) in tags:
        m_,sd_=baseline(tg); st=status_of(aid)
        degr = 1.0 if (st!="ok" and ("VIBR" in tg or "TEMP" in tg)) else 0.0
        base_rows.append((tg, aid, float(m_), float(sd_), float(m_-4*sd_), float(m_+4*sd_),
                          float(rng01(tg,"phase")), 0.5+rng01(tg,"cyc")*3.0, degr))
    base_df = spark.createDataFrame(base_rows, ["tag","asset_id","mean","sd","lo","hi","phase","cycles","degr"])

    end = int(NOW.timestamp())
    def gen_tier(step_sec, span_sec, tier):
        n = span_sec//step_sec
        # rows = tags x timesteps ; timestamp = end - (n-i)*step
        base = base_df.hint("broadcast")
        rng = spark.range(0, n).withColumnRenamed("id","i")
        df = base.crossJoin(rng)
        df = df.withColumn("epoch", (F.lit(end) - (F.lit(n)-F.col("i"))*F.lit(step_sec)))
        df = df.withColumn("ts", F.col("epoch").cast("timestamp"))
        tfrac = (F.col("epoch") % 86400) / 86400.0
        # mean-reverting-ish, parallelizable: sinusoid + seeded gaussian noise + degradation ramp near end
        sinu = F.sin(F.lit(2*math.pi)*(tfrac*F.col("cycles") + F.col("phase")))
        noise = F.randn(SEED) * F.col("sd") * F.lit(0.6)
        ramp_pos = (F.col("i")/F.lit(max(1,n)))
        ramp = F.col("degr") * F.col("sd") * F.lit(3.0) * (ramp_pos*ramp_pos)   # grows toward "now"
        val = F.col("mean") + F.col("sd")*F.lit(0.5)*sinu + noise + ramp
        val = F.greatest(F.col("lo"), F.least(F.col("hi")+F.col("degr")*F.col("sd")*3, val))
        return (df.withColumn("value", F.round(val,3)).withColumn("tier", F.lit(tier))
                  .withColumn("date", F.to_date("ts"))
                  .select("tag","asset_id","ts","value","tier","date"))

    cold = gen_tier(60, HISTORY_DAYS*86400, "cold")     # per-minute, 30d
    hot  = gen_tier(1,  HOT_DAYS*86400,     "hot")       # per-second, 1d
    pievents = cold.unionByName(hot)
    print("PiEvents estimated rows:", (HISTORY_DAYS*1440 + HOT_DAYS*86400) * len(tags))

# CELL ********************  (write Delta tables)
def write(df, name, partition=None):
    dst = f"{TABLES}/{SCHEMA}/{name}"
    w = df.write.format("delta").mode(WRITE_MODE).option("overwriteSchema","true")
    if partition: w = w.partitionBy(*partition)
    w.save(dst)
    print("wrote", f"{SCHEMA}.{name}", "->", dst)

for df,name in [(dim_site,"dim_site"),(dim_unit,"dim_unit"),(dim_asset,"dim_asset"),(dim_tag,"dim_tag"),
                (aakr_health,"aakr_health"),(watchlist,"watchlist"),(anomaly_advisories,"anomaly_advisories"),
                (root_cause,"root_cause"),(predictions_shortterm,"predictions_shortterm"),
                (predictions_longterm,"predictions_longterm"),(fact_work_requests,"fact_work_requests"),
                (daily_narrative,"daily_narrative"),(dim_weather_event,"dim_weather_event"),
                (WeatherForecast,"WeatherForecast"),(fact_asset_exposure,"fact_asset_exposure"),
                (fact_posture,"fact_posture")]:
    write(df, name)
if PIEVENTS_ENABLED:
    write(pievents, "PiEvents", partition=["tier","date"])

# CELL ********************  (validation — counts + referential integrity)
def cnt(name): return spark.read.format("delta").load(f"{TABLES}/{SCHEMA}/{name}").count()
print("=== row counts ===")
for t_ in ["dim_site","dim_unit","dim_asset","dim_tag","aakr_health","watchlist","anomaly_advisories",
           "root_cause","predictions_shortterm","predictions_longterm","fact_work_requests",
           "dim_weather_event","WeatherForecast","fact_asset_exposure","fact_posture"]:
    print(f"  {t_:24s} {cnt(t_):>10,}")
if PIEVENTS_ENABLED: print(f"  {'PiEvents':24s} {cnt('PiEvents'):>12,}")

print("=== referential integrity (should all be 0 orphans) ===")
def orphans(child, ckey, parent, pkey):
    c=spark.read.format("delta").load(f"{TABLES}/{SCHEMA}/{child}"); p=spark.read.format("delta").load(f"{TABLES}/{SCHEMA}/{parent}")
    o=c.join(p, c[ckey]==p[pkey], "left_anti").count(); print(f"  {child}.{ckey} -> {parent}.{pkey}: {o}"); return o
orphans("dim_unit","site_id","dim_site","site_id")
orphans("dim_asset","site_id","dim_site","site_id")
orphans("dim_asset","unit_id","dim_unit","unit_id")
orphans("dim_tag","asset_id","dim_asset","asset_id")
orphans("aakr_health","asset_id","dim_asset","asset_id")
orphans("fact_asset_exposure","asset_id","dim_asset","asset_id")
orphans("fact_asset_exposure","site_id","dim_site","site_id")
print("done.")
